In [ ]:
!pip -q install fastcoref spacy
!python -m spacy download en_core_web_sm -q

In [ ]:
text = ("Rahul bought a laptop yesterday. He said it was very fast. "
        "Priya liked the laptop too, so she bought it for her sister.")
print(text)

In [ ]:
USE_NEURAL = True
try:
    from fastcoref import FCoref
    model = FCoref(device="cpu")
    preds = model.predict(texts=[text])
    clusters = preds[0].get_clusters()          # list of clusters (each = list of mention strings)
    for i, c in enumerate(clusters, 1):
        print(f"Cluster {i}: {c}")
except Exception as e:
    USE_NEURAL = False
    print("fastcoref could not run, using rule-based fallback ->", e)

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")

def simple_resolve(text):
    doc = nlp(text)
    last_person, last_thing, out = None, None, []
    for tok in doc:
        word = tok.text
        if tok.ent_type_ == "PERSON":
            last_person = tok.text
        elif tok.pos_ == "NOUN" and tok.dep_ in ("dobj", "nsubj"):
            last_thing = tok.text
        elif tok.lower_ in ("he", "she") and last_person:
            word = last_person
        elif tok.lower_ == "it" and last_thing:
            word = "the " + last_thing
        out.append(word + tok.whitespace_)
    return "".join(out)

print("Original :", text)
print("Resolved :", simple_resolve(text))